# VHL-specific neosurf-on-neosurf search (CUTOFF = 6)

A targeted variant of the main screen: only VHL structures bound to **3JF (VH032)**,
**4YY (VH101)**, or **6Z3 (VH298)** are used as query targets, searched against the full
seed library, with the neosurface patch cutoff widened to **6 Å** on both target and seed
(`config_vhl_cutoff6.sh`) so patches further from the ligand are sampled.

**Isolation** — everything for this run lives under `data/vhl_cutoff6/` and nothing in the
original `data/masif_search/` or `data/processing/` is touched. Stages 1–2 (prepare input,
preprocess) are **reused** from the main screen (surfaces/descriptors are cutoff-independent),
so this notebook starts at Stage 3.

> SLURM submission is guarded by `SUBMIT_SLURM` (default `False`). Gather/enrich cells only
> read and write CSVs, so they are safe to re-run.

In [ ]:
import os
import sys
import glob
import subprocess
import numpy as np
import pandas as pd

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)
sys.path.insert(0, os.path.join(repo_root, "masif_seed_search/source"))
sys.path.insert(0, os.path.join(repo_root, "scripts/python"))

# ----- This run -----
CONFIG_FILE = "scripts/configs/config_vhl_cutoff6.sh"   # CUTOFF=6 preset
SEED_SITE_CUTOFF = 6.0                                    # must match config CUTOFF (used by enrichment)
N_ARRAY_JOBS = 500
SUBMIT_SLURM = False
VHL_GENE = "VHL"
VHL_LIGANDS = ["3JF", "4YY", "6Z3"]                       # VH032, VH101, VH298

data_dir = os.path.join(repo_root, "data")

# ----- Reused read-only from the main screen (stages 1-2) -----
preprocess_dir = os.path.join(data_dir, "preprocess")
preprocess_ok_csv = os.path.join(data_dir, "processing", "2_masif_preprocess", "df_preprocess_ok.csv")
benchmark_surfaces_dir = os.path.join(preprocess_dir, "data_preparation", "01-benchmark_surfaces")
benchmark_pdbs_dir = os.path.join(preprocess_dir, "data_preparation", "01-benchmark_pdbs")

# ----- Isolated outputs for this run (data/vhl_cutoff6/...) -----
run_root = os.path.join(data_dir, "vhl_cutoff6"); os.makedirs(run_root, exist_ok=True)
masif_search_out_dir = os.path.join(run_root, "masif_search"); os.makedirs(masif_search_out_dir, exist_ok=True)
master_subset_dir = os.path.join(masif_search_out_dir, "subset"); os.makedirs(master_subset_dir, exist_ok=True)
query_targets_list = os.path.join(masif_search_out_dir, "query_targets.txt")

proc_dir = os.path.join(run_root, "processing")
masif_search_proc_dir = os.path.join(proc_dir, "3_masif_search"); os.makedirs(masif_search_proc_dir, exist_ok=True)
enrich_metrics_proc_dir = os.path.join(proc_dir, "4_enrich_metrics"); os.makedirs(enrich_metrics_proc_dir, exist_ok=True)

results_csv = os.path.join(masif_search_proc_dir, "df_results_all.csv")
results_dedup_csv = os.path.join(masif_search_proc_dir, "df_results_dedup.csv")
enriched_all_csv = os.path.join(enrich_metrics_proc_dir, "df_results_all.csv")
enriched_dedup_csv = os.path.join(enrich_metrics_proc_dir, "df_results_dedup.csv")
n_target_sites_csv = os.path.join(enrich_metrics_proc_dir, "n_target_sites.csv")
n_seed_candidates_csv = os.path.join(enrich_metrics_proc_dir, "n_seed_candidates.csv")

print("CONFIG_FILE:", CONFIG_FILE, "| outputs ->", run_root)

## Stage 3 — MaSIF search (VHL targets, CUTOFF=6)

Reuses the shared `df_preprocess_ok.csv`: seeds are the full library; query targets are the VHL 3JF/4YY/6Z3 structures.

In [ ]:
# Seeds = full library; query targets = VHL structures bound to the three ligands.
df_preprocess_ok = pd.read_csv(preprocess_ok_csv)
df_preprocess_ok["seed_id"] = df_preprocess_ok["target"]

seed_ids = df_preprocess_ok.loc[df_preprocess_ok["split"] == "seed", "seed_id"].tolist()

df_targets = df_preprocess_ok[df_preprocess_ok["split"] == "target"]
vhl_targets = df_targets[(df_targets["gene_name"] == VHL_GENE) & (df_targets["ligand_code"].isin(VHL_LIGANDS))]
query_targets = vhl_targets["seed_id"].tolist()

print(f"VHL query targets ({len(query_targets)}): {query_targets}")
print(f"seeds: {len(seed_ids)}")
vhl_targets[["target", "pdb_id", "ligand_code", "uniprot_id", "resolution"]]

In [ ]:
# Submit the search. CONFIG_FILE is forwarded to the array job via sbatch --export so the
# containerized search sources config_vhl_cutoff6.sh (CUTOFF=6). Output is isolated.
def submit_neosurf_search(query_targets, seed_ids, n_array_jobs, masif_search_out_dir,
                          config_file=None, dry_run=True):
    """Write query_targets.txt + seed subset files and submit search_array.sh.

    config_file: path (relative to repo root) sourced by the search via CONFIG_FILE;
    None uses the default scripts/config.sh.
    """
    master_subset_dir = os.path.join(masif_search_out_dir, "subset")
    os.makedirs(master_subset_dir, exist_ok=True)
    query_target_txt = os.path.join(masif_search_out_dir, "query_targets.txt")
    if not isinstance(query_targets, (list, np.ndarray)):
        query_targets = [query_targets]

    if not dry_run:
        with open(query_target_txt, "w") as f:
            for t in query_targets:
                f.write(f"{t}\n")
        for idx, chunk in enumerate(np.array_split(seed_ids, n_array_jobs)):
            with open(os.path.join(master_subset_dir, f"{idx+1}"), "w") as sf:
                for s in chunk:
                    sf.write(f"{s}\n")

    export = "ALL" + (f",CONFIG_FILE={config_file}" if config_file else "")
    cmd = ["sbatch", f"--export={export}", f"--array=1-{n_array_jobs}",
           "scripts/slurm/search_array.sh", query_target_txt, masif_search_out_dir, master_subset_dir]
    print(" ".join(map(str, cmd)))
    if not dry_run:
        subprocess.run(cmd, check=True)


submit_neosurf_search(query_targets, seed_ids, N_ARRAY_JOBS, masif_search_out_dir,
                      config_file=CONFIG_FILE, dry_run=not SUBMIT_SLURM)

## Stage 4 — Gather results & enrich metrics

Same as the main pipeline, but reading/writing the isolated `data/vhl_cutoff6/` paths and
using `SEED_SITE_CUTOFF = 6` for `n_seed_candidates`. Read/write CSVs only.

In [ ]:
# Gather every clustered_matches csv into df_results_all.
csv_files = glob.glob(os.path.join(masif_search_out_dir, "*", "clustered_matches", "*.csv"))
df_results = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"Loaded {len(df_results)} rows from {df_results['target'].nunique()} targets ({len(csv_files)} files).")
df_results.to_csv(results_csv, index=False)
df_results.head()

In [ ]:
# Merge preprocess metadata, deduplicate to one row per (target, matched_protein, cluster_id),
# and add n_match_ligands.
df_results = pd.read_csv(results_csv)
df_preprocess_ok = pd.read_csv(preprocess_ok_csv)

target_meta = df_preprocess_ok[["target", "gene_name"]].rename(columns={"gene_name": "target_gene_name"})
seed_meta = df_preprocess_ok[["target", "uniprot_id", "gene_name", "recommendedName", "mw"]].rename(columns={
    "target": "matched_protein", "uniprot_id": "matched_uniprot_id", "gene_name": "matched_gene_name",
    "recommendedName": "matched_recommendedName", "mw": "matched_mw"})
df_results = df_results.merge(target_meta, on="target", how="left").merge(seed_meta, on="matched_protein", how="left")

df_results_dedup = (df_results.sort_values("score", ascending=False)
                    .drop_duplicates(subset=["target", "matched_protein", "cluster_id"]))

# n_match_ligands: number of unique (target, matched_protein) pairs per (target_gene_name, matched_gene_name).
combo = (df_results_dedup.groupby(["target_gene_name", "matched_gene_name"])
         .apply(lambda x: x.drop_duplicates(subset=["target", "matched_protein"]).shape[0])
         .reset_index(name="n_match_ligands"))
df_results_dedup = df_results_dedup.merge(combo, on=["target_gene_name", "matched_gene_name"], how="left")

df_results_dedup.to_csv(results_dedup_csv, index=False)
print(f"df_results_dedup: {len(df_results_dedup)} rows / {df_results_dedup['target'].nunique()} targets")
df_results_dedup.head()

### Enrich metrics

Adds the following per-row metrics (all computed from committed inputs — no container/SLURM):

| column | definition |
|---|---|
| `target_mw` | molecular weight of the target complex |
| `n_target_sites` | number of anchor sites on the target (`site_*` dirs under `data/masif_search/<target>/`) |
| `n_seed_candidates` | number of seed surface patches near the seed ligand (vertices within `SEED_SITE_CUTOFF` Å of the seed's largest HET residue) |
| `total_n_patches` | `n_target_sites * n_seed_candidates` — search space size for that target×seed pair |
| `cluster_size_patch_normalized` | `cluster_size / total_n_patches` |
| `cluster_size_mw_normalized` | `cluster_size / (target_mw + matched_mw)` |

`n_seed_candidates` is cached per seed in `n_seed_candidates.csv`; newly seen seeds are computed from the surface geometry and appended.

In [ ]:
# Enrichment helpers (pure-python; no container needed).
from Bio.PDB import PDBParser
_ENRICH_PARSER = PDBParser(QUIET=True)


def _read_ply_xyz(path):
    """Return the (N, 3) vertex coordinates of a MaSIF surface .ply (ascii or binary)."""
    with open(path, "rb") as f:
        head = b""
        while b"end_header" not in head:
            head += f.readline()
        text = head.decode("latin1")
        nv = int([l for l in text.splitlines() if l.startswith("element vertex")][0].split()[2])
        fmt = [l for l in text.splitlines() if l.startswith("format")][0].split()[1]
        if fmt == "ascii":
            return np.array([[float(x) for x in f.readline().split()[:3]] for _ in range(nv)])
        tm = {"float": "f4", "double": "f8", "uchar": "u1", "int": "i4", "uint": "u4",
              "char": "i1", "short": "i2", "ushort": "u2"}
        vprops, in_vertex = [], False
        for l in text.splitlines():
            if l.startswith("element vertex"):
                in_vertex = True; continue
            if l.startswith("element") and "vertex" not in l:
                in_vertex = False
            if l.startswith("property") and in_vertex:
                vprops.append(l.split())
        dt = np.dtype([(p[-1], "<" + tm[p[1]]) for p in vprops])
        arr = np.frombuffer(f.read(dt.itemsize * nv), dtype=dt, count=nv)
        return np.stack([arr["x"], arr["y"], arr["z"]], 1)


def _ligand_anchor_coords(pdb_path):
    """Heavy-atom coords of the largest HET residue (matches masif find_ligand_anchor)."""
    st = _ENRICH_PARSER.get_structure("s", pdb_path)
    groups = {}
    for res in st.get_residues():
        if not res.id[0].strip():
            continue
        groups.setdefault((res.parent.id, res.id[1], res.get_resname().strip()), []).append(res)
    best, best_count = None, -1
    for residues in groups.values():
        c = sum(1 for r in residues for a in r if a.element != "H")
        if c > best_count:
            best_count, best = c, residues
    if best is None:
        return None
    return np.array([a.coord for r in best for a in r if a.element != "H"])


def compute_n_seed_candidates(seed_id):
    """# seed surface vertices within SEED_SITE_CUTOFF of the seed's ligand anchor."""
    ply = os.path.join(benchmark_surfaces_dir, f"{seed_id}.ply")
    pdb = os.path.join(benchmark_pdbs_dir, f"{seed_id}.pdb")
    if not (os.path.exists(ply) and os.path.exists(pdb)):
        return np.nan
    anchor = _ligand_anchor_coords(pdb)
    if anchor is None or len(anchor) == 0:
        return np.nan
    V = _read_ply_xyz(ply)
    dmin = np.sqrt(((V[:, None, :] - anchor[None, :, :]) ** 2).sum(-1)).min(1)
    return int((dmin < SEED_SITE_CUTOFF).sum())

In [ ]:
# Compute enriched metrics for both tables and write to 4_enrich_metrics/.
df_preprocess_ok = pd.read_csv(preprocess_ok_csv)
mw = df_preprocess_ok.set_index("target")["mw"]

df_all = pd.read_csv(results_csv)       # raw gathered results
df_dedup = pd.read_csv(results_dedup_csv)  # metadata-merged + deduplicated

# n_target_sites: number of anchor sites per target (site_* dirs).
targets = pd.unique(pd.concat([df_all["target"], df_dedup["target"]]))
n_target_sites = {t: len(glob.glob(os.path.join(masif_search_out_dir, t, "site_*"))) for t in targets}
pd.Series(n_target_sites, name="n_target_sites").to_csv(n_target_sites_csv)

# n_seed_candidates: per seed, cached; compute + append any newly-seen seeds.
if os.path.exists(n_seed_candidates_csv):
    nsc = pd.read_csv(n_seed_candidates_csv, index_col=0)["n_seed_candidates"].to_dict()
else:
    nsc = {}
seeds = pd.unique(pd.concat([df_all["matched_protein"], df_dedup["matched_protein"]]))
new_seeds = [s for s in seeds if s not in nsc]
for s in new_seeds:
    nsc[s] = compute_n_seed_candidates(s)
if new_seeds:
    pd.Series(nsc, name="n_seed_candidates").sort_index().to_csv(n_seed_candidates_csv)
print(f"n_target_sites for {len(n_target_sites)} targets; computed {len(new_seeds)} new seed candidate counts")


def add_enrichment(df, is_dedup):
    df = df.copy()
    orig = list(df.columns)
    if "matched_mw" not in df.columns:
        df["matched_mw"] = df["matched_protein"].map(mw)
    df["target_mw"] = df["target"].map(mw)
    df["n_target_sites"] = df["target"].map(n_target_sites)
    df["n_seed_candidates"] = df["matched_protein"].map(nsc)
    df["total_n_patches"] = df["n_target_sites"] * df["n_seed_candidates"]
    df["cluster_size_patch_normalized"] = df["cluster_size"] / df["total_n_patches"]
    df["cluster_size_mw_normalized"] = df["cluster_size"] / (df["target_mw"] + df["matched_mw"])
    metrics = ["n_target_sites", "n_seed_candidates", "total_n_patches",
               "cluster_size_patch_normalized", "cluster_size_mw_normalized"]
    tail = ["target_mw"] + metrics if is_dedup else ["target_mw", "matched_mw"] + metrics
    return df[orig + tail]


add_enrichment(df_all, is_dedup=False).to_csv(enriched_all_csv, index=False)
df_enriched_dedup = add_enrichment(df_dedup, is_dedup=True)
df_enriched_dedup.to_csv(enriched_dedup_csv, index=False)
print(f"wrote enriched all + dedup ({len(df_enriched_dedup)} rows) to {enrich_metrics_proc_dir}")
df_enriched_dedup.head()